[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S04_numpy_ndarray.ipynb)

# Sesión 04 · El ndarray de NumPy

**Módulo 2: NumPy** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Explicar en qué se diferencia un `ndarray` de una lista y leer sus atributos `shape`, `ndim`, `size` y `dtype`.
2. Crear arrays con `np.array`, `zeros`, `ones`, `arange` y `linspace`.
3. Aplicar funciones matemáticas a un array completo y entender de dónde salen `inf` y `nan`.
4. Comparar arrays elemento a elemento, operar con escalares (broadcasting) y unir arrays con `concatenate`.
5. Resumir datos con `sum`, `mean`, `median`, `min`, `max`, `std`, `argmin` y `argmax`.

## 📋 Qué debes saber antes
Lo del módulo 1: listas, bucles, funciones y comprensiones de listas.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Hoy no hace falta ningún bucle: todo se resuelve con operaciones sobre arrays completos.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos de práctica y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión y las funciones que revisan tus respuestas.
import copy
import hashlib
import math
import statistics

import numpy as np

rng = np.random.default_rng(42)

# ---------- Datos de práctica: ventas de tiendas ----------
dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
ventas_lista = [int(v) for v in rng.integers(800, 5000, size=7)]
meta_diaria = 3000
semana_1 = rng.integers(800, 5000, size=7)
semana_2 = rng.integers(800, 5000, size=7)

unidades = rng.integers(0, 50, size=10)
unidades[3] = 0
ingresos_tienda = np.round(rng.uniform(500, 3000, size=6), 2)
ingresos_tienda[4] = 0.0
clientes_tienda = rng.integers(5, 60, size=6)
clientes_tienda[1] = 0
clientes_tienda[4] = 0
tasas_continuas = np.array([0.0, 0.03, 0.05, 0.08])

# ---------- Datos de práctica: movimientos bancarios ----------
saldo_inicial = round(float(rng.uniform(200, 800)), 2)
montos_mes = np.round(rng.normal(-60, 80, size=30), 2)
montos_mes[0] = round(float(rng.uniform(2500, 3500)), 2)
montos_mes[14] = 0.0
tipo_cambio = 3.75

_NOMBRES = ["dias", "ventas_lista", "meta_diaria", "semana_1", "semana_2", "unidades", "ingresos_tienda",
            "clientes_tienda", "tasas_continuas", "saldo_inicial", "montos_mes", "tipo_cambio"]
_D = copy.deepcopy({k: globals()[k] for k in _NOMBRES})
# Versiones en listas de Python: los verificadores recalculan con bucles, sin NumPy.
_L = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in _D.items()}

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        igual = (np.array_equal(actual, original) if isinstance(original, np.ndarray) else actual == original)
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    lista = _L["ventas_lista"]
    _arr(r, "ventas", lista, "debería contener los mismos valores que `ventas_lista`", tipos="iu")
    r.valor("forma", (len(lista),), tuple, "es la tupla que devuelve el atributo de la forma")
    r.valor("n_dim", 1, int, "cuenta las dimensiones del array")
    r.valor("n_elem", len(lista), int, "cuenta todos los elementos del array")
    r.valor("tipo", str(np.array(_D["ventas_lista"]).dtype), str, "convierte el dtype a texto con `str()`")
    v = r.var("de_vuelta", list)
    if v is not _FALTA:
        if v == lista and all(type(x) is int for x in v):
            r.ok("`de_vuelta` es una lista de Python con los valores correctos.")
        else:
            r.mal("`de_vuelta` debería ser la lista que devuelve el método para convertir un array en lista.")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_lista_x2": "4152dd8442a25c19608edaff98259df5e9c2e72140ea93426a877bca387c65b6",
        "pred_array_x2": "4bfc84ac8fd1e690b2309147b1a4cddd6d7a2714fa6aa501a4f3d95ced8e3158",
        "pred_dtype": "2f8dd469968d20ee46ee0fc632f0166ae995b6d83aa1ecfa34632232c2040b6f",
        "pred_suma_0": "7a20ad6fbd72c40ec9ce42fcce75b2afd01e02ef19725a1956c8b5a8b66eeefc",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    _arr(r, "ceros", [0.0] * 7, "deberían ser siete ceros", tipos="f")
    _arr(r, "unos", [1] * 5, "deberían ser cinco unos", tipos="iu")
    _arr(r, "semanas", list(range(1, 53)), "deberían ser los números de semana del 1 al 52")
    _arr(r, "pares", [2 * i for i in range(11)], "deberían ser los pares del 0 al 20, ambos incluidos")
    _arr(r, "precios_prueba", [10 + 5 * i for i in range(9)], "deberían ser 9 valores igualmente espaciados entre 10 y 50")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_arange_vacio": "25af31a57c2047d854d189042b0ecfb66843c4c19d3dd9ce1403e0be3826a240",
        "pred_incluye_lin": "504fd320c035cf419fcfb0f0cbe7f3b478d10f78fda40022eeb35a86b23bce08",
        "pred_incluye_ara": "98f1c15b97f0aab4934b54ee33b6a9fd148f07ee171d3618e796e5ba5b2cbf3d",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    u = _L["unidades"]
    _arr(r, "raiz", [math.sqrt(x) for x in u], "aplica la raíz cuadrada a cada elemento de `unidades`", tipos="f")
    _arr(r, "log_unidades", [math.log1p(x) for x in u],
         "usa el logaritmo que funciona con ceros (logaritmo de 1 + x)", tipos="f")
    ticket = []
    for a, b in zip(_L["ingresos_tienda"], _L["clientes_tienda"]):
        ticket.append(math.nan if a == 0 and b == 0 else (math.inf if b == 0 else a / b))
    _arr(r, "ticket", ticket, "debería ser `ingresos_tienda` entre `clientes_tienda`, elemento a elemento", tipos="f")
    _arr(r, "ticket_red", [x if not math.isfinite(x) else round(x, 2) for x in ticket],
         "debería ser `ticket` redondeado a 2 decimales", tol=0.0051, tipos="f")
    _arr(r, "factor", [round(math.exp(t), 4) for t in _L["tasas_continuas"]],
         "debería ser la exponencial de cada tasa, redondeada a 4 decimales", tol=0.00006, tipos="f")
    _sin_cambios(r, "unidades", "ingresos_tienda", "clientes_tienda")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_div_cero": "b646d9519248ca84420086ceb9be9992b677d4ea06678cc1aa1d36f938032961",
        "pred_cero_cero": "7cd2a33d8476047b3755295f8b4747a3baea572e22f0e24d21505be7bc3c9844",
        "pred_nan_igual": "98f1c15b97f0aab4934b54ee33b6a9fd148f07ee171d3618e796e5ba5b2cbf3d",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4")
    v, meta = _L["ventas_lista"], _D["meta_diaria"]
    s1, s2 = _L["semana_1"], _L["semana_2"]
    _arr(r, "sobre_meta", [x >= meta for x in v], "cada elemento debe responder si la venta alcanza la meta", tipos="b")
    _arr(r, "con_igv", [round(x * 118 / 100, 2) for x in v], "cada venta con 18 % de IGV, redondeada a 2 decimales", tol=0.0051)
    _arr(r, "en_miles", [x / 1000 for x in v], "cada venta dividida entre 1000")
    _arr(r, "diferencia_meta", [x - meta for x in v], "cada venta menos la meta (puede ser negativa)")
    _arr(r, "dos_semanas", s1 + s2, "deberían ir primero los 7 días de `semana_1` y luego los 7 de `semana_2`")
    _arr(r, "cambio", [b - a for a, b in zip(s1, s2)], "cada día de la semana 2 menos el mismo día de la semana 1")
    _arr(r, "mejoro", [b > a for a, b in zip(s1, s2)], "cada elemento debe responder si la semana 2 vendió más que la 1", tipos="b")
    _sin_cambios(r, "semana_1", "semana_2")
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    v = _L["ventas_lista"]
    _esc(r, "total", math.fsum(v), "suma todas las ventas")
    _esc(r, "media", statistics.fmean(v), "calcula el promedio")
    _esc(r, "mediana", statistics.median(v), "la mediana es el valor del medio con los datos ordenados")
    _esc(r, "minimo", sorted(v)[0], "busca el menor valor")
    _esc(r, "maximo", sorted(v)[-1], "busca el mayor valor")
    _esc(r, "desv", statistics.pstdev(v), "usa la desviación estándar por defecto de NumPy (poblacional)")
    _esc(r, "dia_max", v.index(sorted(v)[-1]), "debería ser la posición del máximo, no su valor")
    _esc(r, "dia_min", v.index(sorted(v)[0]), "debería ser la posición del mínimo, no su valor")
    r.valor("nombre_dia_max", _D["dias"][v.index(sorted(v)[-1])], str, "usa la posición del máximo para buscar en `dias`")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_argmax_empate": "d024f6472cd1381eeddb28a59daaff16528770c069c615713e06c20d6b4498fd",
        "pred_media_bool": "2a0443a6d969186ead6fa44cb991d9e8cae3815e4bba429abdc44d1c2c7527ac",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    m = _L["montos_mes"]
    ingresos = math.fsum(x for x in m if x > 0)
    egresos = math.fsum(x for x in m if x < 0)
    n_egr = len([x for x in m if x < 0])
    _esc(r, "total_ingresos", round(ingresos, 2), "suma solo los montos positivos y redondea a 2 decimales", tol=0.0051)
    v = globals().get("total_egresos")
    if _es_numero(v) and float(v) > 0:
        r.mal("`total_egresos` debería ser negativo: es la suma de los montos negativos.")
    else:
        _esc(r, "total_egresos", round(egresos, 2), "suma solo los montos negativos y redondea a 2 decimales", tol=0.0051)
    _esc(r, "neto", round(math.fsum(m), 2), "suma todos los montos y redondea a 2 decimales", tol=0.0051)
    _esc(r, "n_dias_egreso", n_egr, "cuenta los días con monto negativo (el día con 0 no cuenta)")
    _esc(r, "promedio_egreso", round(egresos / n_egr, 2), "divide el total de egresos entre la cantidad de días con egreso", tol=0.0051)
    _esc(r, "dia_mayor_egreso", m.index(sorted(m)[0]) + 1, "debería ser el número de día (del 1 al 30), no la posición")
    _arr(r, "montos_usd", [round(x / _D["tipo_cambio"], 2) for x in m], "divide cada monto entre `tipo_cambio` y redondea", tol=0.0051)
    _esc(r, "saldo_final", round(_D["saldo_inicial"] + math.fsum(m), 2), "saldo inicial más el neto del mes", tol=0.0051)
    _sin_cambios(r, "montos_mes")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    saldo, saldos = _D["saldo_inicial"], []
    for x in _L["montos_mes"]:
        saldo += x
        saldos.append(saldo)
    _arr(r, "saldo_diario", saldos, "cada elemento debe ser el saldo al cierre de ese día", tol=0.0051)
    _esc(r, "dias_negativos", len([s for s in saldos if s < 0]), "cuenta los días que cerraron con saldo menor que 0")
    r.fin()


print("✅ Setup listo. Datos generados y verificadores cargados.")

### 📦 Tus datos de hoy
Los valores se generan con una semilla fija, así que siempre salen iguales.

In [ ]:
print("🏪 Ventas de tiendas")
print("dias            =", dias)
print("ventas_lista    =", ventas_lista, "(una lista de Python)")
print("meta_diaria     =", meta_diaria)
print("semana_1        =", semana_1)
print("semana_2        =", semana_2)
print("unidades        =", unidades)
print("ingresos_tienda =", ingresos_tienda)
print("clientes_tienda =", clientes_tienda)
print("tasas_continuas =", tasas_continuas)
print()
print("🏦 Movimientos bancarios")
print("saldo_inicial =", saldo_inicial, "| tipo_cambio =", tipo_cambio)
print("montos_mes    =", montos_mes)

---
## 1. De la lista al `ndarray`

### 📘 Concepto
NumPy se importa con el alias `np` (el setup ya lo hizo: `import numpy as np`). Su estructura central es el **`ndarray`**: una tabla de valores **del mismo tipo**, sobre la que las operaciones se aplican a todos los elementos a la vez, sin bucles.

| | Lista | ndarray |
|---|---|---|
| Tipos | puede mezclarlos | todos iguales (si mezclas, NumPy convierte al más general) |
| `* 2` | repite la lista | multiplica cada elemento |
| `+` entre dos | las une | suma elemento a elemento |

Atributos útiles: `a.shape` (forma, como tupla), `a.ndim` (número de dimensiones), `a.size` (total de elementos) y `a.dtype` (tipo de los elementos: `int64`, `float64`, `bool`...). Con `a.tolist()` vuelves a una lista de Python.

In [ ]:
precios_l = [10, 20, 30]
precios_a = np.array(precios_l)

print(precios_l * 2)      # repite
print(precios_a * 2)      # multiplica cada elemento
print(precios_a + precios_a)

print(type(precios_a), precios_a.shape, precios_a.ndim, precios_a.size, precios_a.dtype)
print(np.array([1, 2, 3.5]).dtype)       # un decimal convierte todo a float
print(precios_a.tolist(), type(precios_a.tolist()))

### ✍️ Tu turno · Ejercicio 1: tu primer array
**Parte A.**
1. Crea `ventas`, un array a partir de `ventas_lista`.
2. Guarda sus atributos en `forma`, `n_dim` y `n_elem`.
3. `tipo`: el `dtype` de `ventas` convertido a texto con `str()`.
4. `de_vuelta`: `ventas` convertido otra vez en lista de Python.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_lista_x2` | `len([1, 2, 3] * 2)` | número |
| `pred_array_x2` | `len(np.array([1, 2, 3]) * 2)` | número |
| `pred_dtype` | `str(np.array([1, 2.5]).dtype)` | texto, por ejemplo `"int64"` |
| `pred_suma_0` | el primer elemento de `np.array([1, 2]) + np.array([3, 4])` | número |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Los atributos se escriben sin paréntesis: `ventas.shape`, no `ventas.shape()`. `tolist` sí es un método y lleva paréntesis.
</details>

<details><summary>💡 Pista 2</summary>

`forma` es una tupla, aunque tenga un solo número: `(7,)`. En la parte B, recuerda qué hace `* 2` con una lista y qué hace con un array.
</details>

---
## 2. Crear arrays desde cero

### 📘 Concepto
| Función | Qué crea | Ejemplo |
|---|---|---|
| `np.zeros(n)` | `n` ceros (float) | `np.zeros(3)` → `[0., 0., 0.]` |
| `np.ones(n)` | `n` unos (float) | `np.ones(2)` → `[1., 1.]` |
| `np.arange(inicio, fin, paso)` | como `range`: el `fin` **no** se incluye | `np.arange(0, 10, 3)` → `[0, 3, 6, 9]` |
| `np.linspace(inicio, fin, n)` | `n` valores igualmente espaciados, **incluido** el `fin` | `np.linspace(0, 1, 5)` → `[0., 0.25, 0.5, 0.75, 1.]` |

Casi todas aceptan `dtype=` para elegir el tipo: `np.zeros(3, dtype=int)`.

Regla práctica: `arange` cuando conoces el **paso**; `linspace` cuando conoces **cuántos** valores quieres.

In [ ]:
print(np.zeros(4))
print(np.ones(3, dtype=int))
print(np.arange(5))            # de 0 a 4
print(np.arange(2, 11, 4))     # 2, 6, 10
print(np.linspace(0, 100, 5))  # 5 valores de 0 a 100

### ✍️ Tu turno · Ejercicio 2: arrays de apoyo
**Parte A.** Crea:
1. `ceros`: siete ceros decimales (uno por día, para ir llenando después).
2. `unos`: cinco unos **enteros**.
3. `semanas`: los números de semana del 1 al 52.
4. `pares`: los números pares del 0 al 20, **ambos incluidos**, usando `arange`.
5. `precios_prueba`: 9 precios igualmente espaciados entre 10 y 50, incluidos los dos.

**Parte B · casos borde.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_arange_vacio` | `len(np.arange(5, 5))` |
| `pred_incluye_lin` | ¿`50 in np.linspace(10, 50, 9)`? (`True` o `False`) |
| `pred_incluye_ara` | ¿`20 in np.arange(0, 20, 2)`? (`True` o `False`) |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Con `arange`, el `fin` tiene que ser uno más allá del último valor que quieres.
</details>

<details><summary>💡 Pista 2</summary>

Para `pares`: empieza en 0, paso 2, y un `fin` que deje entrar al 20. Para `precios_prueba` usa `linspace` con 9 puntos.
</details>

---
## 3. Funciones matemáticas, `inf` y `nan`

### 📘 Concepto
Las **ufuncs** de NumPy se aplican a cada elemento: `np.sqrt`, `np.exp`, `np.log` (logaritmo natural), `np.log1p` (logaritmo de `1 + x`) y `np.round(a, decimales)`.

`np.log(0)` da `-inf`: por eso, para datos con ceros (unidades, ventas) se usa `np.log1p`, que en 0 vale 0.

Cuando un array divide entre cero, NumPy **no se detiene**: muestra un `RuntimeWarning` (un aviso, no un error) y pone un valor especial:
- número / 0 → `inf` (infinito, con el signo del número).
- 0 / 0 → `nan` (*not a number*: el resultado no está definido).

`nan` "contamina" todo lo que toca (`nan + 1` es `nan`) y **no es igual a nada, ni a sí mismo**. En la sesión 7 verás cómo detectarlo y limpiarlo.

In [ ]:
valores_ej = np.array([0, 1, 4, 9])
print(np.sqrt(valores_ej))
print(np.log1p(valores_ej))
print(np.round(np.exp(np.array([0, 1])), 3))

numerador_ej = np.array([10.0, 0.0, -3.0])
denominador_ej = np.array([2, 0, 0])
print(numerador_ej / denominador_ej)     # verás un aviso: es normal

### ✍️ Tu turno · Ejercicio 3: transformar y dividir
**Parte A.**
1. `raiz`: la raíz cuadrada de cada elemento de `unidades`.
2. `log_unidades`: el logaritmo de `unidades` usando la función que funciona con ceros.
3. `ticket`: el ticket promedio por tienda, `ingresos_tienda` entre `clientes_tienda`. Algunas tiendas no tuvieron clientes: deja que aparezcan `inf` y `nan`.
4. `ticket_red`: `ticket` redondeado a 2 decimales.
5. `factor`: `e` elevado a cada tasa de `tasas_continuas` (el factor de crecimiento continuo), redondeado a 4 decimales.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_div_cero` | ¿qué muestra NumPy en `np.array([5.0]) / 0`? | texto: `"inf"`, `"nan"` o `"error"` |
| `pred_cero_cero` | ¿y en `np.array([0.0]) / 0`? | texto: `"inf"`, `"nan"` o `"error"` |
| `pred_nan_igual` | ¿`np.nan == np.nan`? | `True` o `False` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Cada punto es una sola línea que aplica una función de NumPy (o un operador) al array completo.
</details>

<details><summary>💡 Pista 2</summary>

`factor` combina dos funciones: primero la exponencial y luego el redondeo. En la parte B, relee qué pasa al dividir entre cero con arrays.
</details>

---
## 4. Comparaciones, broadcasting y `concatenate`

### 📘 Concepto
- **Operar con un escalar** (un solo número) lo aplica a cada elemento: `a * 1.18`, `a - 100`, `a / 1000`. Esto es el **broadcasting**: NumPy "estira" el número para que tenga la forma del array.
- **Operar dos arrays de la misma forma** trabaja posición a posición: `b - a`.
- **Comparar** (`>`, `>=`, `==`, `!=`...) devuelve un array de `bool` del mismo tamaño.
- `np.concatenate([a, b])` une arrays uno detrás de otro. Ojo: `a + b` **suma**, no une.

In [ ]:
precios_ej = np.array([20.0, 55.0, 80.0])
nuevos_ej = np.array([22.0, 50.0, 80.0])

print(precios_ej * 0.9)            # 10 % de descuento a todos
print(nuevos_ej - precios_ej)      # cambio de cada precio
print(precios_ej >= 50)            # array de bool
print(nuevos_ej != precios_ej)
print(np.concatenate([precios_ej, nuevos_ej]))

### ✍️ Tu turno · Ejercicio 4: comparar semanas
Usa el array `ventas` del ejercicio 1:
1. `sobre_meta`: array de `bool`, `True` en los días que alcanzan `meta_diaria`.
2. `con_igv`: cada venta con 18 % de IGV, redondeada a 2 decimales.
3. `en_miles`: cada venta expresada en miles.
4. `diferencia_meta`: cuánto le sobra (positivo) o le falta (negativo) a cada día respecto de la meta.

Con `semana_1` y `semana_2`:

5. `dos_semanas`: las dos semanas seguidas en un solo array de 14 días.
6. `cambio`: cuánto cambió cada día de la semana 2 respecto del mismo día de la semana 1.
7. `mejoro`: array de `bool`, `True` donde la semana 2 vendió más que la 1.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Todo se hace con operadores sobre arrays completos, sin bucles. Cada punto es una línea.
</details>

<details><summary>💡 Pista 2</summary>

`concatenate` recibe **una lista** de arrays: `np.concatenate([x, y])`. Para `cambio` y `mejoro`, el orden de la resta y de la comparación importa: semana 2 frente a semana 1.
</details>

---
## 5. Resumir: agregaciones

### 📘 Concepto
| Función | Qué devuelve |
|---|---|
| `np.sum(a)` o `a.sum()` | suma |
| `np.mean(a)` o `a.mean()` | promedio |
| `np.median(a)` | mediana (el valor del medio al ordenar; resiste mejor los valores extremos) |
| `np.min(a)`, `np.max(a)` | mínimo y máximo |
| `np.std(a)` | desviación estándar **poblacional** (divide entre `n`) |
| `np.argmin(a)`, `np.argmax(a)` | la **posición** del mínimo y del máximo (si hay empate, la primera) |

Con un array de `bool`, `True` cuenta como 1 y `False` como 0: `np.sum` cuenta cuántos `True` hay y `np.mean` da la **proporción**. Lo usarás mucho en la sesión 5.

In [ ]:
visitas_ej = np.array([120, 340, 90, 340, 210])
print(visitas_ej.sum(), visitas_ej.mean(), np.median(visitas_ej))
print(visitas_ej.min(), visitas_ej.max(), round(visitas_ej.std(), 2))
print(np.argmin(visitas_ej), np.argmax(visitas_ej))

dias_ej = ["lun", "mar", "mié", "jue", "vie"]
print("Mejor día:", dias_ej[np.argmax(visitas_ej)])
print(np.sum(visitas_ej > 200), np.mean(visitas_ej > 200))   # cuántos y qué proporción

### ✍️ Tu turno · Ejercicio 5: la semana en una línea por dato
**Parte A.** Con el array `ventas`, calcula `total`, `media`, `mediana`, `minimo`, `maximo` y `desv` (desviación estándar). Luego:
- `dia_max` y `dia_min`: las **posiciones** del día de mayor y de menor venta.
- `nombre_dia_max`: el nombre del día de mayor venta, tomado de `dias`.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_argmax_empate` | `np.argmax([3, 7, 7, 1])` |
| `pred_media_bool` | `np.mean(np.array([True, False, True, True]))` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Cada variable es una llamada a una función de la tabla. La mediana solo existe como `np.median(...)`, no como método.
</details>

<details><summary>💡 Pista 2</summary>

`argmax` devuelve una posición: úsala como índice en la lista `dias`.
</details>

---
## 🏋️ Reto final: el mes en la cuenta
`montos_mes` tiene el movimiento neto de cada día de un mes (30 días; el día 1 está en la posición 0): positivo si entró dinero y negativo si salió. Un día tuvo movimiento 0. Resuelve **sin bucles**:

1. `total_ingresos`: suma de los montos positivos. Pista: multiplicar un array por un array de `bool` deja en 0 los elementos donde hay `False`.
2. `total_egresos`: suma de los montos negativos (será un número negativo).
3. `neto`: suma de todos los montos.
4. `n_dias_egreso`: cuántos días tuvieron monto negativo.
5. `promedio_egreso`: egreso promedio de los días con egreso.
6. `dia_mayor_egreso`: el **número de día** (del 1 al 30) con el egreso más grande.
7. `montos_usd`: los montos en dólares (divide entre `tipo_cambio`), redondeados a 2 decimales.
8. `saldo_final`: `saldo_inicial` más el neto del mes.

Redondea a 2 decimales los puntos 1, 2, 3, 5 y 8. No modifiques `montos_mes`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

`montos_mes > 0` es un array de `bool`. Multiplícalo por `montos_mes` y suma. Para contar, suma directamente el array de `bool`.
</details>

<details><summary>💡 Pista 2</summary>

El egreso más grande es el monto más negativo, o sea, el **mínimo**. `argmin` da una posición que empieza en 0: súmale 1 para obtener el número de día.
</details>

---
## 🚀 Nivel pro (opcional)
1. `saldo_diario`: el saldo al cierre de cada día del mes, en un array de 30 elementos. Investiga `np.cumsum` (suma acumulada) y combínalo con `saldo_inicial`.
2. `dias_negativos`: cuántos días cerraron con saldo menor que 0 (puede ser cero).
3. Sin verificador: compara con `%timeit` cuánto tarda `sum(lista)` frente a `np.sum(array)` con un millón de números (`np.arange(1_000_000)` y su versión `.tolist()`). ¿Cuántas veces más rápido es NumPy?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar qué hace `* 2` con una lista y con un array.
- [ ] Leer `shape`, `ndim`, `size` y `dtype`, y explicar por qué `shape` es una tupla.
- [ ] Elegir entre `arange` y `linspace`, y saber cuál incluye el valor final.
- [ ] Explicar por qué se usa `np.log1p` con datos que tienen ceros.
- [ ] Explicar de dónde salen `inf` y `nan`, y qué tiene de raro `nan`.
- [ ] Operar un array con un escalar y dos arrays de la misma forma.
- [ ] Explicar la diferencia entre `a + b` y `np.concatenate([a, b])`.
- [ ] Explicar la diferencia entre `np.max` y `np.argmax`.
- [ ] Contar y calcular proporciones con `np.sum` y `np.mean` sobre un array de `bool`.

**Próxima sesión (S05):** indexing y filtrado en 1D con máscaras booleanas.